# Lesson 12: OCR for PDFs without a text layer (solution)

**Goal:** find the pages that need OCR, read one with the local OCR model, and try threads.

**Before you start**
- Theory: [lesson 12](../../../docs/lessons/12_ocr.md)
- Python concepts: `concurrent.futures.ThreadPoolExecutor`, `pool.map`, the GIL
- Files used: `fixtures/tiny_scanned.pdf` (the same 3 pages as tiny.pdf, but as images). Offline, no API key, no cost.

Each step: read the short explanation, fill in the `# TODO` cell, run the check cell below it.

## Setup (just run it)

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from docrag.ingest.ocr import RapidOcrEngine, pages_needing_ocr, render_png

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
scanned = (ROOT / "fixtures" / "tiny_scanned.pdf").read_bytes()
normal = (ROOT / "fixtures" / "tiny.pdf").read_bytes()

## Step 1: Which pages need OCR?

`pages_needing_ocr(pdf_bytes)` returns the 1-based pages that have (almost) no text layer but contain an image. Only those get OCR'd, so normal PDFs pay nothing.

In [ ]:
pages = pages_needing_ocr(scanned)

In [ ]:
assert pages == [1, 2, 3], f"All three scanned pages are images, got {pages}"
assert pages_needing_ocr(normal) == [], "tiny.pdf has a text layer: nothing to OCR"
print("OCR needed on pages", pages)
print("OK")

## Step 2: Read page 2 with RapidOCR

`render_png(pdf, page, zoom)` renders one page to a PNG; `RapidOcrEngine().ocr_pages([png, ...])` returns one text per page, lines sorted into reading order. The engine runs PaddleOCR models on ONNX Runtime (GPU if available, else CPU, about 2 s).

In [ ]:
engine = RapidOcrEngine()
png = render_png(scanned, 2, engine.zoom)
text = engine.ocr_pages([png])[0]

In [ ]:
assert "Europe" in text and "45.5" in text, "Page 2 holds the revenue table"
print(text)
print("OK")

## Step 3: Threads

`ThreadPoolExecutor(max_workers=n)` runs a function on n threads; `pool.map(fn, items)` returns results **in input order** (Java's `ExecutorService`, but with a context manager that waits for all tasks). Write `render_all(workers)` that renders every job page at zoom 2.0 and returns the list of PNGs. Then compare 1 vs 4 workers.

In [ ]:
jobs = pages * 4  # 12 renders


def render_all(workers: int) -> list[bytes]:
    with ThreadPoolExecutor(max_workers=workers) as pool:
        return list(pool.map(lambda p: render_png(scanned, p, 2.0), jobs))

In [ ]:
timings = {}
results = {}
for workers in (1, 4):
    start = time.perf_counter()
    results[workers] = render_all(workers)
    timings[workers] = time.perf_counter() - start
assert len(results[1]) == len(jobs), "One PNG per job"
assert results[1] == results[4], "Same output, same order, regardless of threads"
print({w: f"{t:.2f}s" for w, t in timings.items()})
print("A speed-up only appears if the library releases the GIL while it renders.")
print("OK")

## Observation question

Why does OCR only help PDFs without a text layer?

<details>
<summary>Answer</summary>

A text layer is exact: the characters are stored in the PDF, so the parser reads them without errors, with fonts and positions (which give headings and tables). OCR has to *guess* characters from pixels: it can misread digits, merge columns or drop small text, and it costs time (or money, with a vision LLM). So the pipeline OCRs only pages with almost no text but at least one image. In the benchmark, 28 of 134 PDFs were image-only slide decks, and OCR took recall@10 on them from 0 to about 0.8.
</details>

## The real code

- OCR engines: [`src/docrag/ingest/ocr.py`](../../../src/docrag/ingest/ocr.py)
- Vision-LLM OCR: [`src/docrag/ingest/vlm_ocr.py`](../../../src/docrag/ingest/vlm_ocr.py)
- Where OCR runs: [`src/docrag/ingest/indexer.py`](../../../src/docrag/ingest/indexer.py)
- Lesson: [`docs/lessons/12_ocr.md`](../../../docs/lessons/12_ocr.md)